# Os orientadores são especialistas ou generalistas?

**Pergunta 3.** Olhando as bancas de defesa e qualificação do ICMC, cada orientador concentra seus alunos em poucos temas (especialista) ou orienta em várias áreas (generalista)?

**Dados.** Cada linha é uma banca (`data/eventos.csv`). O tema de cada banca (`data/temas_bancas.csv`) foi escolhido por um LLM a partir do **título**, entre os temas oficiais do programa (`data/temas.json`). Entram só os programas com árvore de temas: PPG-CCMC, PPG-Mat e PIPGES. As bancas rotuladas como "Outros" pela LLM ficam de fora.

**Gráficos.** Os dois abaixo são independentes, cada um com seus controles:
1. **Dispersão:** um ponto por orientador, com o nº de bancas e a concentração no tema principal.
2. **Heatmap:** orientador × tema, com a fração das bancas dele em cada tema.

In [ ]:
from pathlib import Path
import unicodedata
import zlib

import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML

RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
ARQ_TEMAS = RAIZ / "data" / "temas_bancas.csv"
ESPACO = HTML('<div style="height:48px"></div>')      # espaço entre os controles e o gráfico
COR = {"PPG-CCMC": "#0072B2", "PIPGES": "#E69F00", "PPG-Mat": "#009E73"}   # paleta que funciona com daltonismo

## 1. Dados

Juntamos as bancas com o tema de cada uma. `folha` é o tema específico (ex: *Processamento de Linguagem Natural*) e `macro` é a área grande que contém esse tema (ex: *Inteligência Computacional*).

In [ ]:
eventos = pd.read_csv(RAIZ / "data" / "eventos.csv")[["id", "orientador"]]
temas = pd.read_csv(ARQ_TEMAS)
bancas = temas.merge(eventos, on="id").rename(columns={"tema_folha": "folha", "tema_macro": "macro"})
n_outros = (bancas.folha == "Outros").sum()
bancas = bancas[bancas.folha != "Outros"]

PROGRAMAS = [p for p in COR if p in set(bancas.programa)]     # na ordem da paleta
print(f"{len(bancas)} bancas de {bancas.orientador.nunique()} orientadores ({n_outros} bancas 'Outros' ficaram de fora)")
bancas[["orientador", "programa", "folha", "macro"]].head()

## 2. Como medimos

Para cada orientador, contamos quantas bancas ele tem em cada tema. Daí sai a **concentração no tema principal**: a % das bancas dele que cai no tema mais frequente dele. Perto de 100% = especialista; baixa = generalista.

Um orientador só entra se tiver pelo menos o **mínimo de bancas** escolhido (com poucas bancas, a medida é instável).

As funções abaixo calculam isso; os gráficos vêm depois.

In [ ]:
def sem_acento(s):
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode().lower()

def casa(nome, busca):
    return busca.strip() != "" and sem_acento(busca.strip()) in sem_acento(nome)

def curto(s, n=22):
    return s if len(s) <= n else s[: n - 1] + "…"

def perfis(programa, minimo, nivel):
    """Uma linha por orientador (perfil) e a tabela orientador x tema com o nº de bancas (conta)."""
    b = bancas if programa == "Todos" else bancas[bancas.programa == programa]
    conta = pd.crosstab(b.orientador, b[nivel])
    n = conta.sum(axis=1)
    conta, n = conta[n >= minimo], n[n >= minimo]
    p = conta.div(n, axis=0)
    perfil = pd.DataFrame({
        "n": n,
        "tema_principal": conta.idxmax(axis=1),
        "concentracao": p.max(axis=1),
    })
    perfil["programa"] = b.groupby("orientador").programa.agg(lambda s: s.value_counts().index[0]).reindex(perfil.index)
    return perfil, conta


def painel(grafico, **controles):
    """Controles + gráfico numa caixa só. Parecido com o interact, mas a cada mudança a saída é
    limpa NA HORA antes de redesenhar (o interact limpa só quando chega a próxima saída)."""
    saida = widgets.Output(layout=widgets.Layout(width="100%"))     # ocupa a largura toda da saída

    def redesenhar(*_):
        saida.clear_output()
        with saida:
            grafico(**{nome: c.value for nome, c in controles.items()})

    for c in controles.values():
        c.observe(redesenhar, names="value")
    display(widgets.VBox([*controles.values(), saida], layout=widgets.Layout(width="100%")))
    redesenhar()

## 3. Gráfico 1: dispersão

Cada ponto é um orientador (cor = programa). No eixo horizontal está o nº de bancas dele; no vertical, a **concentração no tema principal**: mais alto = mais especialista; mais baixo = mais generalista.

- Os pontos têm um pequeno deslocamento horizontal só para não se sobreporem.
- Passe o mouse num ponto para ver o orientador. Clique na legenda para esconder um programa. **Buscar** circula o orientador pelo nome.

In [ ]:
def grafico_dispersao(programa="Todos", nivel="folha", minimo=4, busca=""):
    perfil, _ = perfis(programa, minimo, nivel)
    perfil = perfil.assign(x=[n + ((zlib.crc32(o.encode()) % 1000) / 1000 - 0.5) * 0.36
                              for o, n in zip(perfil.index, perfil.n)])
    display(ESPACO)
    print(f"{len(perfil)} orientadores com pelo menos {minimo} bancas")

    fig = go.Figure()
    for prog in PROGRAMAS:
        q = perfil[perfil.programa == prog]
        if q.empty:
            continue
        texto = [f"<b>{o}</b><br>{r.n} bancas<br>concentração no tema principal: {r.concentracao:.0%}"
                 f"<br>tema principal: {r.tema_principal}" for o, r in q.iterrows()]
        fig.add_trace(go.Scatter(x=q.x, y=q.concentracao, mode="markers", name=prog, text=texto, hoverinfo="text",
                                 marker=dict(color=COR[prog], size=10, opacity=0.85, line=dict(color="white", width=1))))

    a, b = (perfil.n.min(), perfil.n.max()) if len(perfil) else (1, 12)

    achados = perfil[[casa(o, busca) for o in perfil.index]]
    if busca.strip():
        print(f"{len(achados)} encontrado(s) (circulado)" if len(achados) else "nenhum orientador encontrado com esse nome e esse mínimo de bancas")
    if len(achados):
        fig.add_trace(go.Scatter(x=achados.x, y=achados.concentracao, mode="markers+text", text=list(achados.index),
                                 textposition="top center", showlegend=False, hoverinfo="skip",
                                 marker=dict(size=20, color="rgba(0,0,0,0)", line=dict(color="black", width=2))))

    fig.update_xaxes(title_text="nº de bancas do orientador", range=[a - 0.7, b + 0.7], dtick=1, zeroline=False)
    fig.update_yaxes(title_text="concentração no tema principal (% das bancas)", range=[0, 1.04], tickformat=".0%",
                     zeroline=False)
    fig.update_layout(width=1100, height=470, margin=dict(l=70, r=20, t=10, b=120), plot_bgcolor="#fafafa",
                      legend=dict(orientation="h", y=-0.28, x=0))
    fig.show(config={"displayModeBar": False})        # sem a barra de ferramentas do plotly

In [ ]:
painel(grafico_dispersao,
       programa=widgets.Dropdown(options=["Todos", *PROGRAMAS], description="programa"),
       nivel=widgets.Dropdown(options=[("tema específico", "folha"), ("área grande", "macro")], description="nível"),
       minimo=widgets.IntSlider(value=4, min=1, max=12, description="mín. bancas"),
       busca=widgets.Text(placeholder="digite um nome", description="buscar"))

## 4. Gráfico 2: heatmap orientador × tema

Cada coluna é um orientador e cada linha é um tema. A cor é a fração das bancas do orientador naquele tema (azul escuro = alta; **cinza = nenhuma banca**). O nº de bancas de cada um aparece ao passar o mouse.

- Um orientador especialista tem, na coluna dele, **uma célula escura** e o resto cinza; um generalista tem **várias células claras**.
- Em "ordenar por", *mais especialistas primeiro* usa a concentração no tema principal do gráfico 1.
- "Cor mostra" troca a % pelo nº de bancas.
- Passe o mouse numa célula para ver os detalhes. **Buscar** contorna a coluna do orientador.

In [ ]:
ESCALA = [[0, "#e5e7eb"], [0.00001, "#e5e7eb"], [0.0000101, "#c6dbef"], [0.25, "#6baed6"],
          [0.5, "#3182bd"], [0.75, "#08519c"], [1, "#08306b"]]       # zero = cinza; resto, azul claro -> escuro

def grafico_heatmap(programa="Todos", nivel="folha", minimo=4, ordem="esp", cor="pct", busca=""):
    perfil, conta = perfis(programa, minimo, nivel)
    perfil = {
        "esp": perfil.sort_values(["concentracao", "n"], ascending=False),
        "ger": perfil.sort_values(["concentracao", "n"], ascending=[True, False]),
        "n": perfil.sort_values("n", ascending=False),
        "nome": perfil.sort_index(),
    }[ordem]
    conta = conta.loc[perfil.index]
    conta = conta.loc[:, conta.sum() > 0]                     # só as colunas (temas) que alguém usou
    pct = conta.div(perfil.n, axis=0)
    display(ESPACO)
    print(f"{len(perfil)} orientadores com pelo menos {minimo} bancas · cinza = nenhuma banca nesse tema")

    z = (pct if cor == "pct" else conta).values.T                      # linhas = temas, colunas = orientadores
    texto = [[f"<b>{o}</b> ({perfil.n[o]} bancas)<br>tema: {t}<br>" + (f"{pct.loc[o, t]:.0%} das bancas do orientador<br>= {conta.loc[o, t]} de {perfil.n[o]} bancas"
                                                                      if conta.loc[o, t] else "nenhuma banca nesse tema")
              for o in conta.index] for t in conta.columns]
    fig = go.Figure(go.Heatmap(x=list(conta.index), y=list(conta.columns), z=z,
                               text=texto, hoverinfo="text", colorscale=ESCALA, zmin=0,
                               zmax=1 if cor == "pct" else None, xgap=0, ygap=0,
                               colorbar=dict(title="% das bancas" if cor == "pct" else "bancas",
                                             tickformat=".0%" if cor == "pct" else "d",
                                             x=1.01, xanchor="left", y=1, yanchor="top",
                                             lenmode="pixels", len=260, thickness=14)))
    idx = [i for i, o in enumerate(conta.index) if casa(o, busca)]
    if busca.strip():
        print(f"{len(idx)} encontrado(s) (coluna com contorno preto)" if idx else "nenhum orientador encontrado com esse nome e esse mínimo de bancas")
    for i in idx:
        fig.add_shape(type="rect", xref="x", yref="y", x0=i - 0.5, x1=i + 0.5, y0=-0.5, y1=len(conta.columns) - 0.5,
                      line=dict(color="black", width=2.5), layer="above")
    topo = int(0.82 * 5.5 * max((len(o) for o in conta.index), default=0)) + 40      # altura do maior nome girado em 55 graus
    fig.update_xaxes(tickangle=-55, side="top", automargin=True, tickfont=dict(size=9), type="category", fixedrange=True)
    fig.update_yaxes(autorange="reversed", automargin=True, tickfont=dict(size=9), fixedrange=True, tickvals=list(conta.columns),
                     ticktext=[curto(t) for t in conta.columns])
    fig.update_layout( height=max(380, topo + 26 * len(conta.columns) + 40), margin=dict(l=130, r=20, t=topo, b=40), dragmode=False,
                      plot_bgcolor="white")
    fig.show(config={"responsive": True, "displayModeBar": False})        # acompanha a largura da saída

In [ ]:
painel(grafico_heatmap,
       programa=widgets.Dropdown(options=["Todos", *PROGRAMAS], description="programa"),
       nivel=widgets.Dropdown(options=[("tema específico", "folha"), ("área grande", "macro")], description="nível"),
       minimo=widgets.IntSlider(value=4, min=1, max=12, description="mín. bancas"),
       ordem=widgets.Dropdown(options=[("mais especialistas primeiro", "esp"), ("mais generalistas primeiro", "ger"),
                                       ("nº de bancas", "n"), ("nome", "nome")], description="ordem"),
       cor=widgets.Dropdown(options=[("% das bancas do orientador", "pct"), ("nº de bancas", "cnt")], description="cor"),
       busca=widgets.Text(placeholder="digite um nome", description="buscar"))

## 5. Limitações

- **Temas rotulados por LLM, a partir do título.** Títulos curtos ou genéricos podem cair no tema errado. Muita banca de aprendizado de máquina aplicada a um domínio (imagens médicas, EEG...) foi para *Aprendizado de Máquina e Mineração de Dados*, o que infla esse tema e faz os orientadores parecerem mais parecidos do que são.
- **Um tema por banca.** Uma banca que mistura dois assuntos conta só para o principal.
- **Poucas bancas por orientador.** A mediana é de 4. Com 4 bancas, "100% em um tema" pode ser coincidência; por isso existe o mínimo de bancas.
- **Só 3 programas.** MECAI e PROFMAT não têm árvore de temas e ficaram de fora, assim como as bancas "Outros".
- **O período é curto** (as listas semanais do ICMC), então o perfil de um orientador é só o que apareceu nesse período.